# Sound Classification & Detection — ML/DL Model Report

**Author:** 
**Date:** 2026-05-28

End-to-end comparison of ML/DL approaches for audio classification, with brief detection coverage.

## Contents
1. Introduction & Scope
2. Dataset Exploration
3. Audio Feature Analysis
4. Preprocessing & Augmentation
5. Baseline ML Models
6. CNN on Spectrograms
7. Pretrained Transfer Learning
8. Advanced DL (CRNN / Transformers)
9. Sound Event Detection (light)
10. Model Comparison
11. Conclusion & Future Work

## 0. Setup

In [ ]:
import os, glob, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import librosa
import librosa.display
import IPython.display as ipd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = 'data'
FIG_DIR  = 'figures'
MODEL_DIR = 'models'
SR = 22050

## 1. Introduction & Scope

**Problem.** Classify audio clips into N categories (forest sounds / chosen domain).

**Scope.**
- Main: clip-level classification.
- Secondary: sound event detection (frame-level).

**Goal.** Compare classical ML vs deep CNNs vs pretrained transformers on accuracy, training cost, inference latency.

**Dataset.** *(fill in: ESC-50 / UrbanSound8K / BirdCLEF / custom forest dataset)*

## 2. Dataset Exploration

In [ ]:
# TODO: load metadata
# df = pd.read_csv(os.path.join(DATA_DIR, 'meta.csv'))
# df.head()

In [ ]:
# Class distribution
# sns.countplot(data=df, y='label', order=df['label'].value_counts().index)
# plt.title('Class distribution'); plt.show()

In [ ]:
# Duration stats, sample rates
# durations = [librosa.get_duration(path=p) for p in df['path']]
# pd.Series(durations).describe()

In [ ]:
# Listen samples per class
# y, sr = librosa.load(sample_path, sr=SR)
# ipd.Audio(y, rate=sr)

## 3. Audio Feature Analysis

In [ ]:
def plot_features(y, sr, title=''):
    fig, ax = plt.subplots(2, 2, figsize=(12, 8))
    librosa.display.waveshow(y, sr=sr, ax=ax[0,0]); ax[0,0].set_title('Waveform')
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(S, sr=sr, x_axis='time', y_axis='log', ax=ax[0,1]); ax[0,1].set_title('STFT (dB)')
    M = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr), ref=np.max)
    librosa.display.specshow(M, sr=sr, x_axis='time', y_axis='mel', ax=ax[1,0]); ax[1,0].set_title('Mel-spectrogram')
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    librosa.display.specshow(mfcc, x_axis='time', ax=ax[1,1]); ax[1,1].set_title('MFCC')
    plt.suptitle(title); plt.tight_layout(); plt.show()

In [ ]:
# Compare features across classes
# for label in df['label'].unique():
#     p = df[df.label==label].iloc[0]['path']
#     y, sr = librosa.load(p, sr=SR)
#     plot_features(y, sr, title=label)

## 4. Preprocessing & Augmentation

Steps: resample → fixed length (pad/trim) → normalize → augment (noise, time-shift, pitch-shift, SpecAugment) → stratified split.

In [ ]:
def load_fixed(path, sr=SR, dur=4.0):
    y, _ = librosa.load(path, sr=sr, duration=dur)
    target = int(sr*dur)
    if len(y) < target: y = np.pad(y, (0, target-len(y)))
    return y[:target]

def aug_noise(y, snr_db=20):
    rms = np.sqrt(np.mean(y**2))
    noise = np.random.randn(len(y)) * rms / (10**(snr_db/20))
    return y + noise

def aug_time_shift(y, max_shift=0.2):
    s = int(np.random.uniform(-max_shift, max_shift) * len(y))
    return np.roll(y, s)

def aug_pitch(y, sr=SR, steps=2):
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=np.random.uniform(-steps, steps))

def spec_augment(mel, time_mask=20, freq_mask=10):
    m = mel.copy()
    f = np.random.randint(0, freq_mask); f0 = np.random.randint(0, m.shape[0]-f)
    t = np.random.randint(0, time_mask); t0 = np.random.randint(0, m.shape[1]-t)
    m[f0:f0+f, :] = 0; m[:, t0:t0+t] = 0
    return m

In [ ]:
# Stratified split
# X = df['path'].values; y_lab = df['label'].values
# X_tr, X_te, y_tr, y_te = train_test_split(X, y_lab, test_size=0.2, stratify=y_lab, random_state=SEED)
# X_tr, X_val, y_tr, y_val = train_test_split(X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=SEED)

## 5. Baseline ML Models

Features: MFCC mean+std (+ optional chroma, ZCR, centroid). Models: LogReg, SVM, RandomForest, XGBoost.

In [ ]:
def extract_handcrafted(y, sr=SR, n_mfcc=20):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    zcr = librosa.feature.zero_crossing_rate(y)
    cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    feats = np.concatenate([
        mfcc.mean(1), mfcc.std(1),
        [zcr.mean(), zcr.std()],
        [cent.mean(), cent.std()],
        chroma.mean(1), chroma.std(1),
    ])
    return feats

In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import SVC
# from sklearn.ensemble import RandomForestClassifier
# from xgboost import XGBClassifier
#
# baselines = {
#     'LogReg': LogisticRegression(max_iter=1000),
#     'SVM':    SVC(kernel='rbf', probability=True),
#     'RF':     RandomForestClassifier(n_estimators=300, random_state=SEED),
#     'XGB':    XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'),
# }
# results = {}
# for name, mdl in baselines.items():
#     mdl.fit(Xtr_feat, ytr_enc)
#     yp = mdl.predict(Xte_feat)
#     results[name] = {'acc': accuracy_score(yte_enc, yp), 'f1': f1_score(yte_enc, yp, average='macro')}
# pd.DataFrame(results).T

## 6. CNN on Spectrograms

Input: log-mel spectrogram (128 mels). Small CNN baseline + VGG-like.

In [ ]:
# import torch, torch.nn as nn
# from torch.utils.data import Dataset, DataLoader
#
# class MelDataset(Dataset):
#     def __init__(self, paths, labels, augment=False): ...
#     def __len__(self): return len(self.paths)
#     def __getitem__(self, i): ...
#
# class SmallCNN(nn.Module):
#     def __init__(self, n_classes):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
#             nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
#             nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
#             nn.Flatten(), nn.Linear(128, n_classes))
#     def forward(self,x): return self.net(x)

In [ ]:
# Training loop placeholder
# def train_epoch(model, loader, opt, loss_fn, device): ...
# def eval_epoch(model, loader, loss_fn, device): ...
# Track train/val loss + acc per epoch, plot curves.

## 7. Pretrained Transfer Learning

Candidates: **YAMNet**, **VGGish**, **PANNs (CNN14)**, **AST**. Two modes: frozen embeddings + linear head, vs full fine-tune.

In [ ]:
# Example: PANNs CNN14 embeddings
# from panns_inference import AudioTagging
# at = AudioTagging(checkpoint_path=None, device='cuda')
# _, emb = at.inference(audio_batch)  # (B, 2048)
# Fit LogReg / MLP on embeddings.

## 8. Advanced DL — CRNN & Transformers

In [ ]:
# CRNN: CNN feature extractor → BiLSTM → FC
# class CRNN(nn.Module): ...
#
# Transformer: AST / PaSST fine-tune
# from transformers import ASTForAudioClassification, ASTFeatureExtractor
# model = ASTForAudioClassification.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593',
#                                                    num_labels=n_classes, ignore_mismatched_sizes=True)

## 9. Sound Event Detection (light)

Frame-level prediction. Demo with PANNs SED head or simple CRNN with frame-wise output. Metrics: segment-based F1, event-based F1 (`sed_eval`).

In [ ]:
# from panns_inference import SoundEventDetection
# sed = SoundEventDetection(checkpoint_path=None, device='cuda')
# framewise = sed.inference(audio)  # (T, n_classes)
# Threshold + median filter → event intervals.
# Plot framewise heatmap over waveform.

## 10. Model Comparison

In [ ]:
# summary = pd.DataFrame([
#     {'model':'LogReg+MFCC','params':'~1K','train_s':..,'acc':..,'f1':..,'infer_ms':..},
#     {'model':'SVM',        ...},
#     {'model':'XGBoost',    ...},
#     {'model':'SmallCNN',   ...},
#     {'model':'VGG-like',   ...},
#     {'model':'PANNs-emb+LR',...},
#     {'model':'AST-finetune',...},
# ])
# summary.sort_values('f1', ascending=False)

In [ ]:
# Confusion matrix grid + per-class F1 bar chart for best model
# Error analysis: top confused class pairs, listen to misclassified samples.

## 11. Conclusion & Future Work

**Findings.** *(fill in)*
- Best model: …
- Tradeoff: accuracy vs latency vs train cost.
- Hard classes: …

**Limits.** Dataset size, class imbalance, domain shift.

**Future.** Self-supervised pretraining (wav2vec2/HuBERT on domain audio), better SED, deployment quantization.

**References.**
- Piczak (2015) ESC-50.
- Hershey et al. (2017) VGGish / CNN architectures for audio.
- Kong et al. (2020) PANNs.
- Gong et al. (2021) AST.
- Park et al. (2019) SpecAugment.